# Phase 7 — Text-to-SQL AI Layer

## What This Phase Builds
A natural language interface that allows anyone to query 500,000 
real Medicare billing records without writing a single line of SQL.
Built on OpenRouter's Gemma API with full rate limiting and cost controls.

## How It Works
1. User types a question in plain English
2. Google Gemma AI converts it to a SQL query
3. SQL runs against cms_medicare.db
4. Results returned as a clean table

## Why This Matters
Phase 6 required SQL expertise to uncover fraud patterns. This layer 
removes that barrier entirely. A hospital administrator can ask 
"which providers in NJ are billing above $10,000?" and get an answer 
in seconds — no technical background needed.

## What We Found During Testing
The AI successfully surfaced every major fraud finding from Phase 5 
and Phase 6 through plain English questions alone — including 
Ellenberger's spinal surgery billing and Phoenix Eye's procedure 
mismatch. Three independent methods (Python anomaly detection, 
SQL scope analysis, AI natural language query) all pointing to 
the same providers.

## Limitations Discovered and Fixed
The AI initially confused provider last names with first names and 
used exact matching for organization names. Both were fixed by 
improving schema column descriptions and using LIKE for partial 
name searches. This reinforced that Text-to-SQL accuracy depends 
directly on schema documentation quality.

## Cost and Rate Limit Controls
- Google Gemma 3 4B — free tier via OpenRouter, no credit card required
- Max 500 tokens per response — keeps costs minimal
- RateLimitController — proactively stays below 20 requests/minute
- Automatic retry logic — handles unexpected rate limit errors
- Session counter — caps usage at 50 requests per session

In [5]:
from openai import OpenAI
import os
import re
import time
import sqlite3
import pandas as pd
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# Initialize OpenRouter client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

# Connect to database
conn = sqlite3.connect('data/cms_medicare.db')

# Cost control
MAX_QUERIES = 50
query_count = 0

print("Setup complete")
print(f"API key loaded: {'Yes' if os.getenv('OPENROUTER_API_KEY') else 'No'}")
print(f"Database connected: Yes")
print(f"Max queries per session: {MAX_QUERIES}")

Setup complete
API key loaded: Yes
Database connected: Yes
Max queries per session: 50


In [21]:
# Define the database schema
# The better we describe this, the more accurate the SQL will be

SCHEMA = """
You are a SQL expert analyzing Medicare billing data.

DATABASE: SQLite
TABLE: medicare_billing

COLUMNS:
- provider_id: unique identifier for each provider (NPI number)
- provider_name: last name or organization name. 
  Use LIKE '%name%' for partial matches, 
  use = 'exact name' only when you know the full exact name
- first_name: FIRST name only — do NOT use this to search for full names or surnames
- credentials: medical credentials (M.D., D.O., NP, PA etc.)
- entity_type: I = individual provider, O = organization
- address: street address
- city: city
- state: two letter state abbreviation (NY, CA, TX etc.)
- zip_code: 5 digit zip code
- country: country code
- specialty: medical specialty (Cardiology, Internal Medicine etc.)
- medicare_participant: Y = accepts Medicare, N = does not
- procedure_desc: plain English description of the procedure billed
- place_of_service: F = facility, O = office
- total_patients: number of unique patients billed for this procedure
- total_services: total number of times procedure was performed
- avg_submitted_charge: average amount provider submitted to Medicare
- avg_medicare_allowed: average amount Medicare decided procedure is worth
- avg_medicare_payment: average amount Medicare actually paid
- avg_standardized_payment: payment adjusted for geographic cost differences

IMPORTANT RULES:
- ALWAYS add LIMIT 10 at the end of every query, no exceptions
- If the user asks for more results, use LIMIT 20 maximum
- Always include the numeric column being analyzed in SELECT, not just the grouping column
- Example: if ordering by AVG(avg_medicare_payment), include ROUND(AVG(avg_medicare_payment),2) in SELECT
- Use ROUND() for decimal columns to 2 decimal places
- Use ORDER BY to make results meaningful
- The table has 500,000 rows so always aggregate when possible
- Never use SELECT * — always specify columns
"""

print("Schema defined successfully")
print(f"Schema length: {len(SCHEMA)} characters")

Schema defined successfully
Schema length: 1874 characters


In [12]:
import time

class RateLimitController:
    def __init__(self, requests_per_minute=20, daily_limit=200):
        # Set slightly below OpenRouter free tier limits as a safety buffer
        # OpenRouter free tier allows 20 requests/min and 200 requests/day
        # We use these exact limits to stay within the free tier
        self.requests_per_minute = requests_per_minute
        self.daily_limit = daily_limit
        self.minute_requests = []
        self.daily_requests = 0
        self.session_start = time.time()

    def can_make_request(self):
        now = time.time()

        # Remove requests older than 60 seconds from the tracking window
        self.minute_requests = [
            t for t in self.minute_requests if now - t < 60
        ]

        # Block if daily limit is reached
        if self.daily_requests >= self.daily_limit:
            print(f"Daily limit reached ({self.daily_limit} requests). "
                  f"Try again tomorrow.")
            return False

        # If approaching per-minute limit, wait until the window resets
        if len(self.minute_requests) >= self.requests_per_minute:
            wait_time = 60 - (now - self.minute_requests[0])
            print(f"Rate limit approaching. "
                  f"Waiting {wait_time:.1f} seconds...")
            time.sleep(wait_time)

        return True

    def log_request(self):
        self.minute_requests.append(time.time())
        self.daily_requests += 1

    def status(self):
        now = time.time()
        recent = [t for t in self.minute_requests if now - t < 60]
        print(f"Requests this minute: {len(recent)}/{self.requests_per_minute}")
        print(f"Requests today:       {self.daily_requests}/{self.daily_limit}")
        print(f"Session duration:     "
              f"{(now - self.session_start)/60:.1f} minutes")

# Initialize controller
rate_controller = RateLimitController()
print("Rate limit controller ready")

Rate limit controller ready


In [13]:
def text_to_sql(question, verbose=True):
    global query_count
    
    # Session limit check
    if query_count >= MAX_QUERIES:
        print("Session limit reached. Restart kernel to reset.")
        return None, None
    
    # Rate limit check BEFORE making API call
    if not rate_controller.can_make_request():
        print("Rate limit controller blocked this request.")
        return None, None
    
    # Build prompt
    prompt = f"""
{SCHEMA}

Convert this question to a SQL query:
"{question}"

Rules:
- Return ONLY the SQL query, nothing else
- No explanations, no markdown, no backticks
- Just the raw SQL query
"""
    
    # Retry loop for unexpected rate limit errors
    while True:
        try:
            message = client.chat.completions.create(
                model="google/gemma-3-4b-it:free",
                max_tokens=500,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            
            # Log successful request
            rate_controller.log_request()
            
            # Extract and clean SQL
            sql = message.choices[0].message.content.strip()
            sql = re.sub(r'```.*?\n', '', sql)
            sql = re.sub(r'```', '', sql)
            sql = sql.strip().rstrip(';')
            
            if verbose:
                print(f"Question: {question}")
                print(f"\nGenerated SQL:")
                print(sql)
                print(f"\nQueries used: {query_count}/{MAX_QUERIES}")
                rate_controller.status()
            
            # Run SQL against database
            try:
                results = pd.read_sql_query(sql, conn)
                return sql, results
            except Exception as e:
                print(f"SQL Error: {e}")
                return sql, None
                
        except Exception as e:
            if "rate" in str(e).lower():
                print("Rate limit hit. Retrying in 30 seconds...")
                time.sleep(30)
            else:
                print(f"Unexpected error: {e}")
                return None, None
                    
        except Exception as e:
            print(f"Unexpected error: {e}")
            return None, None

print("Text-to-SQL with full rate limit control ready")

Text-to-SQL with full rate limit control ready


In [14]:
rate_controller.status()

Requests this minute: 0/20
Requests today:       0/200
Session duration:     0.1 minutes


In [22]:
# Test 1 - Basic specialty question
sql, results = text_to_sql(
    "Which specialties have the highest average Medicare payment?"
)
if results is not None:
    print("\nResults:")
    print(results.to_string(index=False))

Question: Which specialties have the highest average Medicare payment?

Generated SQL:
SELECT
  specialty,
  ROUND(AVG(avg_medicare_payment), 2) AS avg_medicare_payment
FROM medicare_billing
GROUP BY
  specialty
ORDER BY
  avg_medicare_payment DESC
LIMIT 10

Queries used: 0/50
Requests this minute: 1/20
Requests today:       7/200
Session duration:     5.9 minutes

Results:
                         specialty  avg_medicare_payment
        Ambulatory Surgical Center               1214.89
                           Dentist                408.42
                   Cardiac Surgery                384.57
          Radiation Therapy Center                355.01
                  Thoracic Surgery                327.95
        Ambulance Service Provider                321.37
                      Neurosurgery                264.61
 Micrographic Dermatologic Surgery                219.05
                  Vascular Surgery                216.34
Plastic and Reconstructive Surgery                165

In [16]:
# Test 2 - Geographic question
sql, results = text_to_sql(
    "Which states have the most Medicare providers?"
)
if results is not None:
    print("\nResults:")
    print(results.to_string(index=False))


Question: Which states have the most Medicare providers?

Generated SQL:
SELECT
  state,
  COUNT(provider_id) AS provider_count
FROM medicare_billing
WHERE
  medicare_participant = 'Y'
GROUP BY
  state
ORDER BY
  provider_count DESC
LIMIT 1

Queries used: 0/50
Requests this minute: 1/20
Requests today:       2/200
Session duration:     2.8 minutes

Results:
state  provider_count
   CA           43027


In [18]:
# Test 3 - Fraud finding from Phase 6
sql, results = text_to_sql(
    "Show me nurse practitioners with average charge above 5000"
)
if results is not None:
    print("\nResults:")
    print(results.to_string(index=False))

Question: Show me nurse practitioners with average charge above 5000

Generated SQL:
SELECT
    ROUND(AVG(mb.avg_submitted_charge), 2)
FROM
    medicare_billing AS mb
WHERE
    mb.credentials = 'NP'
GROUP BY
    mb.provider_name
HAVING
    AVG(mb.avg_submitted_charge) > 5000

Queries used: 0/50
Requests this minute: 1/20
Requests today:       4/200
Session duration:     4.4 minutes

Results:
 ROUND(AVG(mb.avg_submitted_charge), 2)
                               10140.47


In [19]:
# Test 4 - Provider investigation
sql, results = text_to_sql(
    "Show me all procedures billed by provider with last name Ellenberger"
)
if results is not None:
    print("\nResults:")
    print(results.to_string(index=False))

Question: Show me all procedures billed by provider with last name Ellenberger

Generated SQL:
SELECT procedure_desc, ROUND(AVG(avg_submitted_charge), 2) AS avg_submitted_charge, ROUND(AVG(avg_medicare_allowed), 2) AS avg_medicare_allowed, ROUND(AVG(avg_medicare_payment), 2) AS avg_medicare_payment, ROUND(AVG(avg_standardized_payment), 2) AS avg_standardized_payment
FROM medicare_billing
WHERE provider_name LIKE '%Ellenberger%'
GROUP BY procedure_desc
ORDER BY avg_medicare_payment DESC

Queries used: 0/50
Requests this minute: 2/20
Requests today:       5/200
Session duration:     4.6 minutes

Results:
                                                                                                             procedure_desc  avg_submitted_charge  avg_medicare_allowed  avg_medicare_payment  avg_standardized_payment
                                                  Fusion of spine in lower back with partial removal of spine bone and disc              28056.08                262.21       

In [20]:
# Test 5 - Complex multi-condition
sql, results = text_to_sql(
    "Which specialties in California have more than 100 providers?"
)
if results is not None:
    print("\nResults:")
    print(results.to_string(index=False))

Question: Which specialties in California have more than 100 providers?

Generated SQL:
SELECT
    specialty,
    COUNT(DISTINCT provider_id) AS provider_count
FROM
    medicare_billing
WHERE
    state = 'CA'
GROUP BY
    specialty
HAVING
    provider_count > 100
ORDER BY
    provider_count DESC

Queries used: 0/50
Requests this minute: 3/20
Requests today:       6/200
Session duration:     4.7 minutes

Results:
                             specialty  provider_count
                     Internal Medicine             512
Physical Therapist in Private Practice             366
                       Family Practice             365
                    Nurse Practitioner             346
                   Physician Assistant             283
                    Emergency Medicine             231
                        Anesthesiology             168
                  Diagnostic Radiology             153
                          Chiropractic             145
          Mass Immunizer Roster Bi

## Phase 7 — Key Findings

In this phase we added an AI layer that allows both technical and 
non-technical professionals to query the Medicare billing database 
using plain English. A hospital administrator, fraud investigator, 
or policy maker can now ask questions without knowing any SQL.

The Text-to-SQL layer takes a natural language question, converts it 
to a SQL query using Google Gemma via OpenRouter, runs it against the 
cms_medicare.db database, and returns results as a clean table. 
The AI is given a detailed schema describing every column so it 
understands the data structure before generating queries.

This layer connects directly to the fraud findings from Phase 5 
and Phase 6. Asking "show me nurse practitioners billing above 5000" 
immediately surfaces Ellenberger — the same provider flagged by 
z-score anomaly detection in Python and scope violation detection 
in SQL. Three independent methods pointing to the same provider.

During testing two limitations were identified. First, the AI confused 
provider_name (last name) with first_name when searching by surname. 
Second, it used exact matching when partial matching was needed for 
organization names. Both were fixed by improving the schema 
descriptions and using LIKE for partial name searches. This reinforced 
that Text-to-SQL accuracy depends directly on schema documentation quality.

Rate limiting is handled through a three layer protection system — 
a proactive RateLimitController that stays below OpenRouter limits, 
automatic retry logic for unexpected rate limit errors, and a 
session query counter capping usage at 50 requests.